In [1]:
import re
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoProcessor
from datasets import load_dataset
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch
import json
from PIL import Image

In [2]:
with open("/root/IC_MLLM_VQA/example/7-12example.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [3]:
# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 加载模型和处理器
model = AutoModelForVision2Seq.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map={"": device},
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

Using device: cuda


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoProcessor
model = PeftModel.from_pretrained(model, "/root/IC_MLLM_VQA/ScienceQA/Lora/qwen2.5vl-Lora7-12_0716/checkpoint-466")
# model = PeftModel.from_pretrained(model, "/root/IC_MLLM_VQA/ScienceQA/Lora/qwen2.5vl-Lora1-6/final_0609_10epoch")

In [5]:

# 构造消息
def build_message(sample):
    content = []
    if sample['image'] is not None:
        content.append({"type": "image", "image": sample['image']})
    
    question_text = f"Question: {sample['question']}\nChoices:\n"
    for idx, choice in enumerate(sample['choices']):
        question_text += f"{chr(65 + idx)}. {choice}\n"
    
    if sample.get("hint"):
        question_text += f"\nHint: {sample['hint']}\n"
    
    question_text += "Please select the correct answer. Then, explain your reasoning in detail and in english. "
    "Make sure your explanation is at least three sentences long, "
    "refers to specific data from the image, and shows your step-by-step logic."
    content.append({"type": "text", "text": question_text})
    
    return [{"role": "user", "content": content}]

# 解析模型输出
import re

def parse_output(output):
    output = output.strip()

    # case 1: "Answer: A Explanation: xxx"
    match = re.search(r"Answer[:：]?\s*([A-D])\b.*?Explanation[:：]?\s*(.+)", output, re.DOTALL)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = match.group(2).strip()
        return answer, explanation

    # case 2: "A Explanation: xxx"
    match = re.match(r"\b([A-D])\s*Explanation[:：]?\s*(.+)", output, re.DOTALL)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = match.group(2).strip()
        return answer, explanation

    # case 3: "A. xxx" or "B: xxx"
    match = re.match(r"\b([A-D])[\.:]\s*(.+)", output, re.DOTALL)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = match.group(2).strip()
        return answer, explanation

    # case 4: only one letter like "C"
    match = re.match(r"^\s*([A-D])\s*$", output)
    if match:
        answer = ord(match.group(1)) - 65
        return answer, ""

    # fallback: try to find first capital letter A-D (unsafe)
    match = re.search(r"\b([A-D])\b", output)
    if match:
        answer = ord(match.group(1)) - 65
        explanation = output[match.end():].strip()
        return answer, explanation

    return -1, ""


# 准备测试数据
N = len(data)
test_dataset = []

for i in range(N):
    sample = data[i]
    try:
        if sample['question'] is None:
            print(f"第 {i} 个样本没有问题，跳过")
            continue

        solution = sample.get("solution", "")
        lecture = sample.get("lecture", "")
        solution_lecture = f"{solution}\n\n{lecture}".strip()

        test_dataset.append({
            "image": sample.get("image", None),
            "question": sample["question"],
            "choices": sample["choices"],
            "answer": sample["answer"],  # 是选项文本
            "hint": sample.get("hint", None),
            "solution_lecture":sample['explanation']
        })
    except Exception as e:
        print(f"跳过第 {i} 个样本，错误：{e}")
        continue

# 初始化工具
rouge = Rouge()
smoothie = SmoothingFunction().method1
vectorizer = CountVectorizer(stop_words="english").fit([s["solution_lecture"] for s in test_dataset])
keywords = set(vectorizer.get_feature_names_out())

# 开始评估
all_records = []

for sample in tqdm(test_dataset[:1429]):
    messages = build_message(sample)
    print (messages)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # image_inputs = process_vision_info(messages)
    image_inputs = [sample["image"]] if sample["image"] else None
    
    image_path = sample["image"]  
    image = Image.open(image_path).convert("RGB")
    image_inputs = [image]

    inputs = processor(text=[text], images=image_inputs, return_tensors="pt", padding=True).to(device)
    generated_ids = model.generate(**inputs, max_new_tokens=512)
    
    generated_ids_trimmed = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    
    output = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    
    if len(output) > 0:
        output_str = output[0]
        pred_answer, pred_explanation = parse_output(output_str)
    else:
        pred_answer, pred_explanation = -1, ""
    
    # pred_answer, pred_explanation = parse_output(output)
    true_answer = sample["answer"]

    # BLEU-1 和 BLEU-4
    reference = sample["solution_lecture"].split()
    hypothesis = pred_explanation.split()
    bleu1 = sentence_bleu([reference], hypothesis, weights=(1, 0, 0, 0), smoothing_function=smoothie)
    bleu4 = sentence_bleu([reference], hypothesis, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie)

    # ROUGE-L
    try:
        rouge_score = rouge.get_scores(pred_explanation, sample["solution_lecture"])[0]["rouge-l"]["f"]
    except:
        rouge_score = 0.0

    # 关键词重叠
    gt_tokens = set(sample["solution_lecture"].lower().split())
    pred_tokens = set(pred_explanation.lower().split())
    overlap = len(gt_tokens & pred_tokens & keywords)
    keyword_score = overlap / max(len(gt_tokens & keywords), 1)

    all_records.append({
        "Question": sample["question"],
        "Choices": "\n".join(sample["choices"]),
        "True Answer": true_answer,
        "Predicted Answer": pred_answer,
        "Predicted Answer Text": sample["choices"][pred_answer] if 0 <= pred_answer < len(sample["choices"]) else "N/A",
        "Model Output": output,
        "Model Explanation": pred_explanation,
        "Reference Solution": sample["solution_lecture"],
        "BLEU-1": bleu1,
        "BLEU-4": bleu4,
        "ROUGE-L": rouge_score,
        "Keyword Overlap": keyword_score
    })

# 结果表格
results_df = pd.DataFrame(all_records)

# 汇总评估
acc = accuracy_score(results_df["True Answer"], results_df["Predicted Answer"])
f1 = f1_score(results_df["True Answer"], results_df["Predicted Answer"], average='macro')
avg_bleu1 = results_df["BLEU-1"].mean()
avg_bleu4 = results_df["BLEU-4"].mean()
avg_rouge = results_df["ROUGE-L"].mean()
avg_keyword_overlap = results_df["Keyword Overlap"].mean()

# 打印汇总结果
print("\nSummary Metrics:")
print(f"Accuracy: {acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Avg BLEU-1: {avg_bleu1:.4f}")
print(f"Avg BLEU-4: {avg_bleu4:.4f}")
print(f"Avg ROUGE-L: {avg_rouge:.4f}")
print(f"Avg Keyword Overlap: {avg_keyword_overlap:.4f}")

# 保存为 CSV
results_df.to_csv("7-12example.csv", index=False)
print("\n✅ 结果已保存为 model_evaluation_results.csv")


  0%|          | 0/1 [00:00<?, ?it/s]

[{'role': 'user', 'content': [{'type': 'image', 'image': '7-12.png'}, {'type': 'text', 'text': 'Question: Compare the average kinetic energies of the particles in each sample. Which sample has the higher temperature?\nChoices:\nA. neither; the samples have the same temperature\nB. sample B\nC. sample A\n\nHint: The diagrams below show two pure samples of gas in identical closed, rigid containers. Each colored ball represents one gas particle. Both samples have the same number of particles.\nPlease select the correct answer. Then, explain your reasoning in detail and in english. '}]}]


100%|██████████| 1/1 [00:34<00:00, 34.38s/it]


Summary Metrics:
Accuracy: 1.0000
F1 Score: 1.0000
Avg BLEU-1: 0.9161
Avg BLEU-4: 0.8776
Avg ROUGE-L: 0.9195
Avg Keyword Overlap: 0.9375

✅ 结果已保存为 model_evaluation_results.csv
